# 04 — Entraînement Cloud (Google Colab GPU)
**NightVision Gen — Entraînement DDPM conditionnel depuis zéro (Modèles A, B, C)**

Ce notebook est conçu pour être exécuté sur **Google Colab avec accélérateur GPU**.
Il récupère le code depuis le dépôt GitHub, sauvegarde les checkpoints sur **Google Drive** de manière persistante, et permet de sélectionner la variante à entraîner sans toucher au code source.

---
### Plan d'exécution :
1. Vérification de l'environnement GPU (NVIDIA T4 / V100 / A100).
2. Montage de Google Drive pour la persistance des poids.
3. Clonage du dépôt GitHub et installation des dépendances.
4. Téléchargement ou accès aux données LOL-v2 et ExDark (décompression automatique).
5. Sélection de la variante : **Modèle A**, **Modèle B** ou **Modèle C**.
6. Configuration et redirection vers Google Drive.
7. Préparation des DataLoaders LOL-v2 avec patchs 128x128.
8. Lancement de l'entraînement avec le `Trainer` modulaire.
9. Visualisation des courbes de perte et des aperçus visuels.
10. Sauvegarde et synchronisation sur Google Drive.

In [ ]:
# 1. Vérification de l'environnement GPU
!nvidia-smi
import torch
print("PyTorch Version :", torch.__version__)
print("CUDA Disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Détecté     :", torch.cuda.get_device_name(0))
    print("Mémoire VRAM    :", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "Go")
else:
    print("⚠️ Aucun GPU détecté. Activez l'accélérateur GPU dans Exécution > Modifier le type d'exécution.")

In [ ]:
# 2. Montage de Google Drive — un seul dossier pour les datasets et les résultats
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')

def _is_project_folder(name: str) -> bool:
    key = name.lower().replace('_', '-').replace(' ', '')
    return key in {'nightvision-gen', 'nightvisiongen'}

def _contains_datasets(folder: Path) -> bool:
    markers = (
        'LOLv2', 'ExDark',
        'data/LOLv2', 'data/ExDark',
        'data/raw/LOLv2', 'data/raw/ExDark',
    )
    return any((folder / rel).exists() for rel in markers)

candidates = []
if DRIVE_ROOT.exists():
    candidates = [p for p in DRIVE_ROOT.iterdir() if p.is_dir() and _is_project_folder(p.name)]

with_data = [p for p in candidates if _contains_datasets(p)]
pool = with_data or candidates
if pool:
    # Préfère le dossier qui contient déjà LOLv2 / ExDark, puis le nom exact nightvision-gen.
    GDRIVE_DIR = sorted(pool, key=lambda p: (p.name != 'nightvision-gen', len(p.name)))[0]
else:
    GDRIVE_DIR = DRIVE_ROOT / 'nightvision-gen'
    GDRIVE_DIR.mkdir(parents=True, exist_ok=True)

duplicates = [p for p in candidates if p.resolve() != GDRIVE_DIR.resolve()]
print('✅ Dossier unique du projet :', GDRIVE_DIR)
print('   Datasets, checkpoints, logs et images générées restent dans ce dossier.')
if duplicates:
    print('⚠️ Dossiers en double ignorés. Déplacez leur contenu dans le dossier unique, puis supprimez-les :')
    for dup in duplicates:
        print('   -', dup)

In [ ]:
# 3. Clone du dépôt GitHub Public et installation du package
import os, sys
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/HASSANDADDA-04/nightvision-gen.git"

%cd /content

repo_dir = Path("/content/nightvision-gen")
if not repo_dir.exists():
    print(f"📥 Clonage du dépôt public {GITHUB_REPO_URL}...")
    !git clone $GITHUB_REPO_URL
else:
    print("🔄 Mise à jour du dépôt avec les derniers commits...")
    %cd /content/nightvision-gen
    !git pull
    %cd /content

%cd /content/nightvision-gen

# Installation des dépendances et installation physique du package
print("📦 Installation des dépendances Colab...")
!pip install -r requirements-colab.txt
print("📦 Installation standard du package nightvision...")
!pip install . --no-build-isolation --no-deps

# Nettoyage et invalidation des caches d'import Python
import importlib
sys.modules.pop("nightvision", None)
importlib.invalidate_caches()

# Enregistrement explicite dans sys.path pour un accès direct immédiat
src_dir = str(Path("/content/nightvision-gen/src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

try:
    import nightvision
    print("✅ Package nightvision installé et importé avec succès sur Colab !")
except ModuleNotFoundError:
    print("
⚠️ Note Colab : Pour finaliser l'enregistrement dans Python 3.13 :")
    print("👉 Allez dans le menu : Exécution > Redémarrer la session (Runtime > Restart session)")
    print("👉 Puis relancez cette cellule : l'import sera immédiat !")

In [ ]:
# 4. Décompression et préparation des datasets depuis Google Drive vers le SSD rapide de Colab
import os, sys, shutil, zipfile
from pathlib import Path

DATA_RAW = Path("/content/nightvision-gen/data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
Path("/content/nightvision-gen/artifacts/checkpoints").mkdir(parents=True, exist_ok=True)
Path("/content/nightvision-gen/artifacts/logs").mkdir(parents=True, exist_ok=True)
Path("/content/nightvision-gen/artifacts/generated_images").mkdir(parents=True, exist_ok=True)

drive_root = Path("/content/drive/MyDrive")

def find_zip_on_drive(pattern: str) -> Path | None:
    # Recherche intelligente d'une archive zip sur tout Google Drive
    if GDRIVE_DIR.exists():
        for p in GDRIVE_DIR.rglob("*.zip"):
            if pattern.lower() in p.name.lower():
                return p
    # 2. Vérifier à la racine de Mon Drive
    for p in drive_root.glob("*.zip"):
        if pattern.lower() in p.name.lower():
            return p
    # 3. Recherche élargie dans les sous-dossiers de Mon Drive (profondeur 2)
    for folder in drive_root.iterdir():
        if folder.is_dir() and not folder.name.startswith("."):
            try:
                for p in folder.glob("*.zip"):
                    if pattern.lower() in p.name.lower():
                        return p
            except OSError:
                continue
    return None

lol_target = DATA_RAW / "LOLv2"
target_test_file = lol_target / "Real_captured" / "Train" / "Normal" / "normal00567.png"

# Vérification : si les images ne sont pas encore sur le SSD Colab, on décompresse le ZIP
if not target_test_file.exists():
    print("🔍 Recherche de l'archive LOLv2.zip sur Google Drive...")
    lol_zip = find_zip_on_drive("lol")
    
    if lol_zip is None:
        print("
" + "="*70)
        print("❌ ERREUR : Impossible de localiser 'LOLv2.zip' sur votre Google Drive !")
        print("="*70)
        print(f"Dossiers scannés :")
        print(f"  - {GDRIVE_DIR}")
        print(f"  - {drive_root}")
        print("
👉 ACTION REQUISE :")
        print("   Vérifiez que votre fichier 'LOLv2.zip' est bien déposé dans votre Google Drive,")
        print(f"   de préférence dans le dossier du projet : {GDRIVE_DIR}")
        print("="*70 + "
")
        raise FileNotFoundError("LOLv2.zip introuvable sur Google Drive. Téléversez le fichier puis relancez.")

    print(f"📦 Archive trouvée : {lol_zip}")
    print("⚡ Décompression en cours vers le SSD local temporaire de Colab...")
    # Décompression rapide via la commande système unzip
    !unzip -q -o "{str(lol_zip)}" -d "{str(DATA_RAW)}"
    
    # Correction automatique de toute imbrication (ex: LOLv2/LOLv2 -> LOLv2)
    nested = lol_target / "LOLv2"
    if nested.exists() and nested.is_dir():
        for item in nested.iterdir():
            shutil.move(str(item), str(lol_target / item.name))
        try:
            nested.rmdir()
        except Exception:
            pass

    # Si le ZIP a extrait directement 'Real_captured' sans dossier 'LOLv2'
    flat_real = DATA_RAW / "Real_captured"
    if flat_real.exists() and not (lol_target / "Real_captured").exists():
        lol_target.mkdir(parents=True, exist_ok=True)
        shutil.move(str(flat_real), str(lol_target / "Real_captured"))

# Décompression optionnelle pour ExDark
exdark_target = DATA_RAW / "ExDark"
if not (exdark_target / "Bicycle").exists():
    exdark_zip = find_zip_on_drive("exdark")
    if exdark_zip:
        print(f"📦 Archive ExDark trouvée : {exdark_zip.name}")
        !unzip -q -o "{str(exdark_zip)}" -d "{str(DATA_RAW)}"
        nested_ex = exdark_target / "ExDark"
        if nested_ex.exists() and nested_ex.is_dir():
            for item in nested_ex.iterdir():
                shutil.move(str(item), str(exdark_target / item.name))
            try:
                nested_ex.rmdir()
            except Exception:
                pass

# Bilan et validation formelle de la présence des fichiers
train_low_count = len(list((lol_target / "Real_captured" / "Train" / "Low").glob("*.png")))
train_norm_count = len(list((lol_target / "Real_captured" / "Train" / "Normal").glob("*.png")))
test_low_count = len(list((lol_target / "Real_captured" / "Test" / "Low").glob("*.png")))

print("
" + "="*55)
print("📊 BILAN DU DATASET LOL-v2 SUR LE SSD COLAB :")
print(f"  • Images sombres (Train Low)     : {train_low_count} fichiers")
print(f"  • Images normales (Train Normal) : {train_norm_count} fichiers")
print(f"  • Images de test réel (Test Low) : {test_low_count} fichiers")

if target_test_file.exists():
    print(f"  • Vérification : ✅ normal00567.png est bien présent !")
    print("="*55)
    print("🎉 Toutes les images sont prêtes pour l'entraînement !")
else:
    print(f"  • ⚠️ Attention : {target_test_file} est introuvable.")
    print("="*55)
    raise FileNotFoundError("Les images LOL-v2 n'ont pas pu être vérifiées sur le SSD local.")

In [ ]:
# 5. Sélection de la variante du modèle à entraîner
# @title Configuration de l'Expérience { run: "auto" }
MODEL_VARIANT = "A" # @param ["A", "B", "C"]
EPOCHS = 100 # @param {type:"integer"}
BATCH_SIZE = 16 # @param {type:"integer"}
LEARNING_RATE = 1e-4 # @param {type:"number"}
LAMBDA_EDGE = 0.05 # @param {type:"number"}
RESUME_TRAINING = False # @param {type:"boolean"}

print(f"🎯 Variante sélectionnée : Modèle {MODEL_VARIANT}")
print(f"   Époques : {EPOCHS} | Batch size : {BATCH_SIZE} | LR : {LEARNING_RATE}")
if MODEL_VARIANT == "C":
    print(f"   Poids de la perte de contours (lambda_edge) : {LAMBDA_EDGE}")

In [ ]:
# 6. Chargement de la configuration YAML correspondante
from nightvision.utils.config import load_config

config_file = f"configs/train_{MODEL_VARIANT}.yaml"
cfg = load_config(config_file)

# Mise à jour avec les paramètres du notebook
cfg["training"]["epochs"] = EPOCHS
cfg["training"]["batch_size"] = BATCH_SIZE
cfg["training"]["learning_rate"] = LEARNING_RATE
if MODEL_VARIANT == "C":
    cfg["loss"]["lambda_edge"] = LAMBDA_EDGE

# Redirection de la sauvegarde vers Google Drive pour éviter toute perte en cas de déconnexion
drive_ckpt_dir = GDRIVE_DIR / f"checkpoints/model_{MODEL_VARIANT}"
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
cfg["paths"]["checkpoint_dir"] = str(drive_ckpt_dir)

drive_log_dir = GDRIVE_DIR / f"logs/model_{MODEL_VARIANT}"
drive_log_dir.mkdir(parents=True, exist_ok=True)
cfg["paths"]["log_dir"] = str(drive_log_dir)

drive_gen_dir = GDRIVE_DIR / f"generated_images/model_{MODEL_VARIANT}"
drive_gen_dir.mkdir(parents=True, exist_ok=True)
cfg["paths"]["generated_dir"] = str(drive_gen_dir)

print("Configuration prête.")
print("  - Checkpoints :", drive_ckpt_dir)
print("  - Logs CSV    :", drive_log_dir)
print("  - Images gen  :", drive_gen_dir)

In [ ]:
# 7. Préparation des DataLoaders LOL-v2
from torch.utils.data import DataLoader
from nightvision.data.split_manager import LOLv2SplitManager
from nightvision.data.dataset import LOLv2PatchDataset, SyntheticPairDataset

compute_retinex = MODEL_VARIANT in ("B", "C")
manifest_path = Path("data/processed/lol_v2_splits.json")
lol_root = Path("data/raw/LOLv2")

print("Découverte des paires réellement présentes dans", lol_root)
lol_mgr = LOLv2SplitManager(root_dir=lol_root, seed=42)
disk_splits = lol_mgr.create_splits()
disk_by_id = {}
for split_pairs in disk_splits.values():
    for pair in split_pairs:
        disk_by_id[pair.pair_id] = pair

train_pairs = []
val_pairs = []
used_ids = set()

if manifest_path.exists() and disk_by_id:
    print("Alignement sur le manifeste :", manifest_path)
    manifest = LOLv2SplitManager.load_split_manifest(manifest_path, root_dir=lol_root)
    missing = 0
    for split_name, bucket in (("train", train_pairs), ("val", val_pairs)):
        for item in manifest.get(split_name, []):
            pair = disk_by_id.get(item.pair_id)
            if pair is None or not (Path(pair.low_path).is_file() and Path(pair.normal_path).is_file()):
                missing += 1
                continue
            bucket.append(pair)
            used_ids.add(pair.pair_id)
    if missing:
        print(f"⚠️ {missing} paires du manifeste ignorées : image sombre ou référence absente.")
    extra_train = [p for p in disk_splits.get("train", []) if p.pair_id not in used_ids]
    extra_val = [p for p in disk_splits.get("val", []) if p.pair_id not in used_ids]
    if extra_train or extra_val:
        print(f"ℹ️ {len(extra_train) + len(extra_val)} paires trouvées sur le disque et ajoutées.")
        train_pairs.extend(extra_train)
        val_pairs.extend(extra_val)
else:
    train_pairs = list(disk_splits.get("train", []))
    val_pairs = list(disk_splits.get("val", []))

def _both_exist(pair) -> bool:
    return Path(pair.low_path).is_file() and Path(pair.normal_path).is_file()

train_pairs = [p for p in train_pairs if _both_exist(p)]
val_pairs = [p for p in val_pairs if _both_exist(p)]

if train_pairs:
    print(f"✅ Dataset réel LOL-v2 opérationnel : {len(train_pairs)} paires train, {len(val_pairs)} paires val.")
    train_ds = LOLv2PatchDataset(train_pairs, patch_size=128, patches_per_image=4, is_train=True, compute_retinex=compute_retinex)
    val_ds = LOLv2PatchDataset(val_pairs, patch_size=128, patches_per_image=2, is_train=False, compute_retinex=compute_retinex)
else:
    print("⚠️ Aucune paire LOL-v2 complète (Low + Normal). Utilisation du dataset synthétique.")
    print("   Vérifiez data/raw/LOLv2/Real_captured/Train/Low et Train/Normal.")
    train_ds = SyntheticPairDataset(32, image_size=128, compute_retinex=compute_retinex)
    val_ds = SyntheticPairDataset(8, image_size=128, compute_retinex=compute_retinex)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"DataLoaders prêts : {len(train_loader)} batches par époque.")

In [ ]:
# 8. Initialisation du Trainer et Lancement de l'Entraînement
from nightvision.training.trainer import Trainer

trainer = Trainer(config=cfg, train_loader=train_loader, val_loader=val_loader)

# Reprise éventuelle depuis un checkpoint existant
latest_ckpt = drive_ckpt_dir / "checkpoint_latest.pt"
if RESUME_TRAINING and latest_ckpt.exists():
    print("🔄 Reprise de l'entraînement depuis :", latest_ckpt)
    trainer.resume_from_checkpoint(latest_ckpt)

# Lancement de l'entraînement
print(f"🚀 Démarrage de l'entraînement du Modèle {MODEL_VARIANT} depuis zéro...")
trainer.train()

In [ ]:
# 9. Visualisation des Courbes d'Apprentissage et du Dernier Aperçu
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

log_dir_path = Path(cfg["paths"]["log_dir"])

# Chargement robuste : tentative prioritaire via JSON (structure parfaite), repli sur CSV sécurisé
json_files = list(log_dir_path.glob("*_history.json")) + list(Path("artifacts/logs").rglob("*_history.json"))
csv_files = list(log_dir_path.glob("*_metrics.csv")) + list(Path("artifacts/logs").rglob("*_metrics.csv"))

df = None
if json_files and json_files[0].exists():
    try:
        with open(json_files[0], "r", encoding="utf-8") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
    except Exception:
        pass

if df is None and csv_files and csv_files[0].exists():
    try:
        df = pd.read_csv(csv_files[0], on_bad_lines="skip")
    except Exception as e:
        print("Erreur lecture CSV :", e)

if df is not None and not df.empty:
    plt.figure(figsize=(10, 4))
    if "epoch" in df.columns and "train_loss" in df.columns:
        plt.plot(df["epoch"], df["train_loss"], label="Perte Entraînement (Train)", color="#3b82f6", lw=2)
    if "epoch" in df.columns and "val_loss" in df.columns:
        val_df = df.dropna(subset=["val_loss"])
        if not val_df.empty:
            plt.plot(val_df["epoch"], val_df["val_loss"], label="Perte Validation (Val)", color="#ef4444", lw=2, marker="o")
    plt.xlabel("Époque")
    plt.ylabel("Perte")
    plt.title(f"Historique d'Entraînement — Modèle {MODEL_VARIANT}")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("Fichier de métriques en attente de génération.")

# Aperçu de la dernière validation
gen_dir_path = Path(cfg["paths"]["generated_dir"])
preview_imgs = sorted(list(gen_dir_path.glob("val_epoch_*.png")) + list(Path("artifacts/generated_images").rglob("val_epoch_*.png")))
if preview_imgs:
    last_preview = preview_imgs[-1]
    print(f"🖼️ Dernier aperçu visuel généré : {last_preview.name}")
    plt.figure(figsize=(12, 4))
    plt.imshow(Image.open(last_preview))
    plt.axis("off")
    plt.title(f"Aperçu Restauration — {last_preview.stem}")
    plt.tight_layout()
    plt.show()

In [ ]:
# 10. Synchronisation finale sur Google Drive
!cp -r artifacts/logs/* {str(GDRIVE_DIR)}/logs/ 2>/dev/null || true
!cp -r artifacts/generated_images/* {str(GDRIVE_DIR)}/generated_images/ 2>/dev/null || true
print("✅ Checkpoints, logs et aperçus synchronisés sur Google Drive :", GDRIVE_DIR)